# Aggregated event-selection histograms

Loads `merged_histdata.pkl` produced by `scripts/event_selection_aggregate.py` and draws overlays with **`overlay_hists_from_histdata`** (same renderer as the batch aggregator).

**Setup:** point `MERGED_PKL` at your plots output directory (the file is written next to the PNGs).

In [ ]:
from pathlib import Path
import sys

_cwd = Path.cwd().resolve()
CAFPYANA_ROOT = None
for _p in [_cwd, *_cwd.parents]:
    if (_p / "pyanalib").is_dir() and (_p / "analysis_village").is_dir():
        CAFPYANA_ROOT = _p
        break
if CAFPYANA_ROOT is None:
    raise RuntimeError("Could not find cafpyana repo root (need pyanalib/ + analysis_village/). cd to repo or notebooks/.")
sys.path.insert(0, str(CAFPYANA_ROOT))
print("CAFPYANA_ROOT =", CAFPYANA_ROOT)

In [ ]:
%matplotlib inline
import pickle
import matplotlib.pyplot as plt

from analysis_village.numucc_1p0pi.event_selection_pipeline_def import build_pipeline
from analysis_village.numucc_1p0pi.selection_framework import ChunkRunner
from analysis_village.numucc_1p0pi.utils import overlay_hists_from_histdata

_style = CAFPYANA_ROOT / "analysis_village" / "numucc_1p0pi" / "presentation.mplstyle"
if not _style.is_file():
    _style = CAFPYANA_ROOT / "analysis_village" / "numucc_1p0pi" / "notebooks" / "presentation.mplstyle"
try:
    plt.style.use(str(_style))
except Exception as e:
    print("mplstyle:", e)

In [ ]:
# --- edit this ---
# MERGED_PKL = CAFPYANA_ROOT / "analysis_village" / "numucc_1p0pi" / "notebooks" / "merged_histdata.pkl"  # placeholder; override
MERGED_PKL = "/path/to/your/plots_dir/merged_histdata.pkl"

# Cosmic handling (must match what you want for the physics plot)
COSMIC_ESTIMATE = "intime"   # "intime" | "offbeam"
SHOW_COSMIC_MODEL_UNCERTAINTY = True

In [ ]:
def load_merged_archive(path: Path):
    path = Path(path).expanduser().resolve()
    with open(path, "rb") as f:
        blob = pickle.load(f)
    merged = blob["merged"]
    pot_str = blob.get("pot_str", "")
    data_pot = blob.get("data_pot")
    print("keys in archive:", sorted(blob.keys()))
    print("n_hist panels:", len(merged["histdata"]))
    print("pot_str:", pot_str, " data_pot:", data_pot)
    if "exposure_scales" in blob and blob["exposure_scales"]:
        print("exposure_scales:", blob["exposure_scales"])
    return blob, merged, pot_str


def build_spec_lookup():
    pipeline = build_pipeline()
    spec_lookup = {}
    for stage in pipeline:
        for ps in stage.plots:
            key = (stage.key, ChunkRunner.plot_key(stage.key, ps))
            spec_lookup[key] = ps
    return spec_lookup


def plot_overlay_from_merged(
    hist_key,
    merged,
    pot_str,
    spec_lookup,
    *,
    cosmic_estimate=COSMIC_ESTIMATE,
    show_cosmic_model_unc=SHOW_COSMIC_MODEL_UNCERTAINTY,
    save_fig=False,
    save_name=None,
    fig_title_suffix="",
):
    """hist_key: tuple (stage_key, plot_key) e.g. from merged['histdata'].keys()"""
    hd = merged["histdata"][hist_key]
    ps = spec_lookup.get(hist_key)
    if ps is None:
        raise KeyError(f"No PlotSpec for {hist_key}; pipeline may have changed since this pickle was built.")

    if ps.plot_label_template is not None:
        plot_labels = [
            ps.plot_label_template[0],
            ps.plot_label_template[1].replace("{pot}", pot_str),
            (ps.plot_label_template[2].replace("{pot}", pot_str) if len(ps.plot_label_template) >= 3 else ""),
        ]
    else:
        plot_labels = [
            ps.var_config.var_labels[0],
            f"Events / Bin (POT={pot_str})",
            "",
        ]
    if fig_title_suffix:
        plot_labels = [plot_labels[0], plot_labels[1], plot_labels[2] + fig_title_suffix]

    kwargs = dict(ps.save_kwargs)
    kwargs.setdefault("ratio", False)
    kwargs.setdefault("plot", True)
    kwargs.setdefault("save_fig", save_fig)
    kwargs.setdefault("save_name", save_name)
    kwargs["plot_labels"] = plot_labels
    kwargs.setdefault("cosmic_estimate", cosmic_estimate)
    kwargs.setdefault("show_cosmic_model_unc", show_cosmic_model_unc)

    return overlay_hists_from_histdata(hd, var_config=ps.var_config, **kwargs)


def list_hist_keys(merged, prefix=None):
    keys = sorted(merged["histdata"].keys(), key=lambda k: (k[0], k[1]))
    if prefix:
        keys = [k for k in keys if k[0].startswith(prefix) or prefix in k[1]]
    for k in keys:
        print(k)
    return keys

In [ ]:
blob, merged, pot_str = load_merged_archive(MERGED_PKL)
spec_lookup = build_spec_lookup()

In [ ]:
# List available histograms (stage_key, plot_key)
keys = list_hist_keys(merged)
# Example filter:
# list_hist_keys(merged, prefix="2prong-mup")

In [ ]:
# Pick one key from the printed list, e.g. a final-stage topology plot:
_cand = [
    k for k in merged["histdata"].keys()
    if k[0] == "2prong-mup" and "topology" in k[1] and "muon_momentum" in k[1]
]
if not _cand:
    _cand = sorted(merged["histdata"].keys(), key=lambda x: (x[0], x[1]))
example_key = sorted(_cand, key=lambda x: (x[0], x[1]))[0]
print("Plotting", example_key)
out = plot_overlay_from_merged(example_key, merged, pot_str, spec_lookup)
plt.show()

### Batch smoke test
Uncomment to redraw every panel (can be slow).

In [ ]:
# for hk in sorted(merged["histdata"].keys(), key=lambda x: (x[0], x[1])):
#     try:
#         plot_overlay_from_merged(hk, merged, pot_str, spec_lookup, save_fig=False)
#         plt.show()
#     except Exception as e:
#         print("SKIP", hk, e)